## ▶️ Step 0 — set up the notebook (run this first!)

This notebook runs in **Google Colab**. The first cell installs what's needed,
downloads the camp toolbox, and connects the data from Google Drive.

**Before you run it**, make sure you've opened the shared camp Drive folder and
clicked **"Add shortcut to Drive"** (put the shortcut in *My Drive*) — that's how
the notebook finds the data file. Then run the cell below and click **Connect** on
the Drive pop-up. Wait for **✅ Setup complete**, then run the rest top to bottom.


In [ ]:
#@title ▶️ Run me first — set up the notebook  { display-mode: "form" }
# Press the ▶ button. (Double-click the title to see the code.)
import os, sys, glob

print("1/3  installing mne ...")
get_ipython().system('pip install -q "mne==1.10.1"')

print("2/3  downloading the camp toolbox ...")
get_ipython().system('wget -q -O camp_utils.py https://raw.githubusercontent.com/anarghya-das/decoding-the-brain-camp/main/camp_utils.py')

print("3/3  connecting Google Drive for the data ...")
from google.colab import drive
drive.mount("/content/drive")
hits = sorted(glob.glob("/content/drive/MyDrive/**/synapse_preprocessed.pkl", recursive=True))
assert hits, (
    "Could not find synapse_preprocessed.pkl in your Drive.\n"
    "Open the shared camp folder, click 'Add shortcut to Drive', put the shortcut "
    "in 'My Drive', then run this cell again."
)
os.environ["CAMP_DATA_PATH"] = hits[0]
os.environ["CAMP_OUTPUT_DIR"] = "/content/drive/MyDrive/DecodingBrain_outputs"
print(f"\n\u2705 Setup complete. Using data at: {hits[0]}")
print("Your figures will be saved to Drive > DecodingBrain_outputs.")


# Week 2 · Day 8 — Statistical Testing

We keep seeing differences between groups. But how do we know a difference is
**real** and not just luck from a small sample? That's what a **statistical
test** answers. Today: the **Mann–Whitney U test** and the famous (and famously
misunderstood) **p-value**.

### By the end of this notebook you will be able to
1. State a null hypothesis
2. Run a Mann–Whitney U test on two groups
3. Interpret a p-value correctly (and know what it does *not* mean)
4. Understand why we chose this test for this dataset

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import camp_utils as cu

features = pd.read_csv(cu.save_path("features_table.csv"))
print("Loaded features:", features.shape)

## 1. The logic of a test
We start by *assuming there is no difference* — this is the **null hypothesis**:

> **H₀:** EXP and CTRL have the same gamma response (any difference is chance).

A test asks: *if H₀ were true, how surprising is the difference we actually saw?*
The answer is the **p-value** — the probability of seeing a difference this big
(or bigger) just by chance.

- **small p (< 0.05)** → our result would be surprising under "no difference",
  so we have evidence the groups really differ.
- **large p** → the difference could easily be chance; not convincing.

## 2. Why Mann–Whitney U (not a t-test)?
The famous t-test assumes the data are roughly bell-shaped and you have a decent
sample size. With only **18 vs 10** subjects and possible outliers, that's risky.

The **Mann–Whitney U test** is a *non-parametric* test: instead of using the raw
values, it ranks everyone and asks whether one group tends to rank higher. It
doesn't assume a bell curve — much safer for small EEG studies. (This is the
exact test the SYNAPSE paper uses.)

## 3. Run it
`scipy.stats.mannwhitneyu` does the work. We pull the two groups' values, drop
missing ones, and call it.

In [ ]:
def get_values(features, feature, group):
    """The non-missing values of one feature for one group ('EXP' or 'CTRL')."""
    vals = features.loc[features["group"] == group, feature].values
    return vals[~np.isnan(vals)]

exp_vals = get_values(features, "let_gamma", "EXP")
ctrl_vals = get_values(features, "let_gamma", "CTRL")

u_stat, p_value = stats.mannwhitneyu(exp_vals, ctrl_vals, alternative="two-sided")

print(f"EXP  mean = {exp_vals.mean():+.2f} dB  (n={len(exp_vals)})")
print(f"CTRL mean = {ctrl_vals.mean():+.2f} dB  (n={len(ctrl_vals)})")
print(f"\nMann-Whitney U = {u_stat:.1f}")
print(f"p-value        = {p_value:.4f}")
print("Significant at 0.05?", "YES ✅" if p_value < 0.05 else "no")

## 4. Interpreting the p-value — read carefully!
A p-value is one of the most misused numbers in science. Get this right:

✅ **p = 0.03 means:** *if there were truly no group difference, we'd see a gap
this big only 3% of the time.*

❌ It does **NOT** mean "97% chance the groups differ."
❌ It does **NOT** tell you the difference is **big** or **important** — only
whether it's distinguishable from chance.
❌ p > 0.05 does **NOT** prove the groups are the same — maybe we just need more
people.

That's why next notebook adds **effect size**: *how big* is the difference,
regardless of the p-value.

### ✏️ Your turn #1 — write a tester
Fill in this function so it returns the p-value for any feature.

In [ ]:
def test_feature(features, feature):
    """Return the Mann-Whitney p-value comparing EXP vs CTRL for one feature."""
    exp = get_values(features, feature, "EXP")
    ctrl = get_values(features, feature, "CTRL")
    # TODO: run mannwhitneyu and return only the p-value
    # hint: stats.mannwhitneyu(exp, ctrl, alternative="two-sided") returns (U, p)
    return None

p = test_feature(features, "let_gamma")
cu.check(p is not None and abs(p - p_value) < 1e-9,
         f"Your tester returns p = {p:.4f} — correct!" if p else "",
         "Return the second value from stats.mannwhitneyu(...).")

## 5. Test ALL the features at once
Now the power of having a function: loop over every feature and test it.

In [ ]:
feature_cols = [c for c in features.columns if c not in ("subject", "group")]

results = []
for feat in feature_cols:
    p = test_feature(features, feat)
    exp = get_values(features, feat, "EXP")
    ctrl = get_values(features, feat, "CTRL")
    results.append({
        "feature": feat,
        "exp_mean": exp.mean(),
        "ctrl_mean": ctrl.mean(),
        "p_value": p,
    })

results = pd.DataFrame(results).sort_values("p_value")
results["significant"] = results["p_value"] < 0.05
print(results.round(4).to_string(index=False))

### ✏️ Your turn #2 — count and reflect
How many features came out "significant" (p < 0.05)?

In [ ]:
n_significant = None   # TODO: count rows where results["significant"] is True
# hint: results["significant"].sum()

cu.check(n_significant is not None and n_significant == int(results["significant"].sum()),
         f"{n_significant} features were significant at p < 0.05.",
         "Use results['significant'].sum().")

## 6. A trap you just walked into 🪤
We just ran **~21 separate tests**. Here's the problem: if you run 20 tests at
the 5% level and *nothing* is real, you still expect about **1 false positive by
pure chance** (20 × 0.05 = 1). The more tests you run, the more "significant"
results you get for free.

This is the **multiple-comparisons problem**, and it's a huge deal in brain
research where people test thousands of things. The fix — the **FDR correction**
— is the very next notebook.

## 🎯 Wrap-up
You ran a proper non-parametric test, interpreted a p-value honestly, and
discovered the multiple-comparisons trap. Save your results table — Notebook 08
will correct it.

In [ ]:
results.to_csv(cu.save_path("stats_results.csv"), index=False)
print("Saved to outputs/stats_results.csv")

➡️ **Next:** Notebook 08 — effect sizes (how *big*?) and the FDR correction
(which results survive?).